# Day 36 — Evaluation + Grad-CAM: where does it look?

Status: COMPLETE. Consolidated metrics (finetuned wins) + Grad-CAM overlays for
a true positive, true negative, and false positive from test, with a
quantitative focus check (central heat mass). 12/12 tests pass.

In [1]:
import json
from pathlib import Path

m = json.loads(Path("../models/finetune_metrics.json").read_text())
print(f"final model (finetuned): accuracy {m['accuracy']} | ROC-AUC {m['roc_auc']} | "
      f"PR-AUC {m['pr_auc']}")
print(f"F1 {m['f1']} (P {m['precision']}, R {m['recall']}) — 4 missed pneumonias "
      "out of 390")
print("progression: scratch 0.855 -> frozen 0.958 -> finetuned 0.962 ROC-AUC")

final model (finetuned): accuracy 0.8686 | ROC-AUC 0.9622 | PR-AUC 0.9714
F1 0.9040 (P 0.8319, R 0.9897) — 4 missed pneumonias out of 390
progression: scratch 0.855 -> frozen 0.958 -> finetuned 0.962 ROC-AUC


In [2]:
g = json.loads(Path("../models/gradcam_check.json").read_text())
print(f"TP (proba {g['tp']['predicted_proba']}): "
      f"central mass {g['tp']['cam_central_mass']} — focus sits mid-lung, as hoped.")
print(f"TN (proba {g['tn']['predicted_proba']}): "
      f"central mass {g['tn']['cam_central_mass']} — diffuse, no focal driver (correct).")
print(f"FP (proba {g['fp']['predicted_proba']}!): "
      f"central mass {g['fp']['cam_central_mass']} — ~3/4 of heat on borders/corners.")
print("Read: the method works AND indicts the model — its most confident mistake")
print("attends to peripheral artifacts, the textbook shortcut pattern. Overlays:")
print("docs/gradcam_tp.png, docs/gradcam_tn.png, docs/gradcam_fp.png")

TP (proba 0.978): central mass 0.632 — focus sits mid-lung, as hoped.
TN (proba 0.178): central mass 0.479 — diffuse, no focal driver (correct).
FP (proba 0.991!): central mass 0.259 — ~3/4 of heat on borders/corners.
Read: the method works AND indicts the model — its most confident mistake
attends to peripheral artifacts, the textbook shortcut pattern. Overlays:
docs/gradcam_tp.png, docs/gradcam_tn.png, docs/gradcam_fp.png


## Findings

1. **Grad-CAM is a test, and the model half-failed it.** TP focus is central;
   the 0.99-confident FP stares at the periphery. High accuracy coexists with
   artifact reliance — without this check we'd never know.
2. **Two implementation traps caught**: forward hooks fire under `no_grad`
   (guard the grad-hook registration), and `requires_grad` flags come from the
   constructor, not the checkpoint (re-enable for attribution).
3. **FP review is now a Day-40 README item**: show the peripheral-focus overlay
   as the lead limitation, not buried.